In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [37]:
df = pd.read_csv('../data/concat/standardized_dataset.csv')

dfPlug = pd.read_csv('../data/concat/plug_only_labeled.csv')

In [38]:
#df = dfPlug.copy()

In [39]:
df.head()

,Time,Temperature TS inlet (Mean),Temperature TS outlet (Mean),Tank temperature (Mean),Bypass temperature (Mean),Pump outlet pressure (Mean),TS inlet pressure (Mean),TS outlet pressure (Mean),Differential pressure (Mean),Total pressure drop,Flow rate (Mean),reading_interval_s,elapsed_runtime_s,Relative_DP,Plug_Label,Run_ID,Is_Plug_File
0,2021-09-01 10:59:21.171,-1.159118,-0.732404,-1.481301,22.072086,0.138025,0.012297,0.000270,0.012027,0.125728,401.648971,1.501,3052.824,0.964510,0,No-Plug_Run_1,0
1,2021-09-01 10:59:22.670,-1.139775,-0.723658,-1.475450,22.074648,0.138058,0.012873,0.000206,0.012667,0.125185,397.179186,1.499,3054.323,1.015835,0,No-Plug_Run_1,0
2,2021-09-01 10:59:24.169,-1.138374,-0.711486,-1.496726,22.079202,0.139616,0.013366,0.000178,0.013188,0.126250,400.550950,1.499,3055.822,1.057616,0,No-Plug_Run_1,0
3,2021-09-01 10:59:25.670,-1.136470,-0.701184,-1.502549,22.079958,0.139463,0.013517,0.000284,0.013233,0.125946,403.609025,1.501,3057.323,1.061225,0,No-Plug_Run_1,0
4,2021-09-01 10:59:27.169,-1.133120,-0.689945,-1.457588,22.076837,0.139763,0.013472,0.000413,0.013059,0.126291,403.325107,1.499,3058.822,1.047271,0,No-Plug_Run_1,0


In [40]:
all_runs = df['Run_ID'].unique()

np.random.seed(42) 
test_runs = np.random.choice(all_runs, size=int(len(all_runs) * 0), replace=False)
train_runs = [run for run in all_runs if run not in test_runs]

train_df = df[df['Run_ID'].isin(train_runs)].copy()
test_df = df[df['Run_ID'].isin(test_runs)].copy()

print(f"Training on {len(train_runs)} runs. Testing on {len(test_runs)} runs.")

Training on 17 runs. Testing on 0 runs.


In [41]:
features = [
    'Temperature TS inlet (Mean)', 'Temperature TS outlet (Mean)', 
    'Pump outlet pressure (Mean)', 'Flow rate (Mean)', 
    'Relative_DP'
]

In [42]:
scaler = StandardScaler()
scaler.fit(train_df[features])

X_train_scaled = pd.DataFrame(scaler.transform(train_df[features]), columns=features, index=train_df.index)
#X_test_scaled = pd.DataFrame(scaler.transform(test_df[features]), columns=features, index=test_df.index)

y_train = train_df['Plug_Label']

In [43]:
groups = train_df['Run_ID']
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(class_weight='balanced', kernel='rbf')
}

In [ ]:
results_data = []

for name, model in models.items():
    
    gkf = GroupKFold(n_splits=11)
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_scaled, y_train, groups), 1):
        X_train_cv, X_val_cv = X_train_scaled.iloc[train_idx], X_train_scaled.iloc[val_idx]
        y_train_cv, y_val_cv = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        val_run_id = groups.iloc[val_idx].unique()[0]
        
        model.fit(X_train_cv, y_train_cv)
        preds = model.predict(X_val_cv)
      
        f1 = f1_score(y_val_cv, preds, zero_division=0)
        acc = accuracy_score(y_val_cv, preds)
        
        results_data.append({
            "Model": name,
            "Validation_Run": val_run_id, 
            "F1-Score": f1,
            "Accuracy": acc
        })

df_results = pd.DataFrame(results_data)

pivot_table = df_results.pivot(index=["Validation_Run"], 
                               columns="Model", 
                               values=["F1-Score", "Accuracy"])

avg_values = pivot_table.mean()
pivot_table.loc[('AVERAGE'), :] = avg_values

display(pivot_table.round(4).sort_values(by=('Validation_Run'), ascending=False))

/opt/anaconda3/envs/Dat255/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/envs/Dat255/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/envs/Dat255/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/envs/Dat255/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/Dat255/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/Dat255/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encounter

F1-Score                                            \
Model          Gradient Boosting Logistic Regression Random Forest     SVM   
Validation_Run                                                               
Plug_Run_6                0.9477              0.9599        0.9716  0.9617   
Plug_Run_2                0.8566              0.8675        0.0366  0.9584   
Plug_Run_11               0.7543              0.8251        0.0000  0.9480   
Plug_Run_1                0.0000              0.0870        0.0000  0.9756   
No-Plug_Run_6             0.0000              0.0000        0.0000  0.0000   
No-Plug_Run_5             0.0000              0.0000        0.0000  0.0000   
No-Plug_Run_4             0.0000              0.0000        0.0000  0.0000   
No-Plug_Run_3             0.8810              0.8499        0.8360  0.8947   
No-Plug_Run_2             0.0000              0.0000        0.0000  0.0000   
No-Plug_Run_1             0.9980              0.8258        0.0000  0.9980   
AVERAGE                   0.4438              0.4415        0.1844  0.5736   

                        Accuracy                                            
Model          Gradient Boosting Logistic Regression Random Forest     SVM  
Validation_Run                                                              
Plug_Run_6                0.9328              0.9481        0.9626  0.9488  
Plug_Run_2                0.8443              0.8500        0.3950  0.9496  
Plug_Run_11               0.7384              0.7366        0.3486  0.9354  
Plug_Run_1                0.9877              0.9883        0.9877  0.9994  
No-Plug_Run_6             0.9582              0.3838        0.9986  0.9815  
No-Plug_Run_5             1.0000              0.7882        1.0000  0.6294  
No-Plug_Run_4             0.9948              0.6785        0.9742  0.9988  
No-Plug_Run_3             0.9487              0.9273        0.9254  0.9554  
No-Plug_Run_2             1.0000              1.0000        1.0000  0.5992  
No-Plug_Run_1             0.9994              0.9527        0.8406  0.9994  
AVERAGE                   0.9404              0.8253        0.8433  0.8997

In [45]:
run_counts = df.groupby('Run_ID')['Plug_Label'].value_counts().unstack(fill_value=0)
run_counts.columns = ['No_Plug_Rows', 'Plug_Rows']
run_counts['Total_Rows'] = run_counts['No_Plug_Rows'] + run_counts['Plug_Rows']
run_counts['Plug_Percentage'] = (run_counts['Plug_Rows'] / run_counts['Total_Rows'] * 100).round(2)

print("--- Distribution per Run ID ---")
display(run_counts.sort_values(by='Run_ID', ascending=False))

--- Distribution per Run ID ---


,No_Plug_Rows,Plug_Rows,Total_Rows,Plug_Percentage
Run_ID,,,,
Plug_Run_9,159,484,643,75.27
Plug_Run_8,142,378,520,72.69
Plug_Run_7,270,7,277,2.53
Plug_Run_6,299,0,299,0.00
Plug_Run_5,967,93,1060,8.77
Plug_Run_4,268,246,514,47.86
Plug_Run_3,47,263,310,84.84
Plug_Run_2,30,97,127,76.38
Plug_Run_11,998,1865,2863,65.14
